<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module3_Files/notebooks/Inside_the_ConvNet_Activations_and_GradCAM.ipynb)

# Inside the ConvNet: how did it decide?

We already trained a cats-vs-dogs model. But a prediction like *"dog, 98% sure"* is a black box. This notebook opens it up in three steps:

1. **Raw image goes in** - a 150x150x3 picture.
2. **The magic: watch the feature maps light up** - we peek at the activations inside each convolution layer. Early layers fire on edges and outlines; deep layers get sparse and abstract.
3. **How did we get the final answer?** - **Grad-CAM** paints a heatmap of *where the model looked* to make its call. We'll do a **confident hit** (right, for a sensible reason) and a **confident miss** (wrong - and the heatmap shows why).

## Setup: load our trained model + some images

In [ ]:
import os, zipfile, urllib.request, glob
import numpy as np
import tensorflow as tf, keras
from keras.utils import load_img, img_to_array
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

# our trained cats/dogs model (saved earlier, hosted in the course repo)
model_url = "https://github.com/drdave-teaching/OPIM5509Files/raw/main/OPIM5509_Module3_Files/data/cats_and_dogs_small_2.keras"
urllib.request.urlretrieve(model_url, "/content/cats_and_dogs_small_2.keras")
model = keras.models.load_model("/content/cats_and_dogs_small_2.keras")

# the cats/dogs images, so we have real pictures to feed in
zip_url = "https://github.com/drdave-teaching/OPIM5509Files/raw/main/OPIM5509_Module3_Files/data/dogs-cats-sep.zip"
urllib.request.urlretrieve(zip_url, "/content/dogs-cats-sep.zip")
with zipfile.ZipFile("/content/dogs-cats-sep.zip") as z: z.extractall("/content/catdog")
test_dir = "/content/catdog/dogs-cats-sep/test"   # 0 = cat, 1 = dog (alphabetical); sigmoid = P(dog)

model.summary()

## 1) Raw image goes in

The model only ever sees a 150x150x3 array of numbers. Here's one test image, loaded exactly the way the model expects (resized to 150x150, pixels scaled to 0-1).

In [ ]:
def load(path):
    """Load an image the way the model expects: 150x150, scaled to 0-1."""
    return img_to_array(load_img(path, target_size=(150, 150))) / 255.

demo = load(test_dir + "/dogs/dog.1769.jpg")
plt.imshow(demo); plt.title("raw image in  (150 x 150 x 3)"); plt.axis('off'); plt.show()

## 2) The magic: watch the feature maps light up

A convolution layer outputs one **feature map per filter** - each one lights up where its pattern appears. We build a helper model that returns the activations after *every* conv and pool layer, then run our image through it.

(Keras 3 note: for a `Sequential` model you can't grab `layer.output` directly, so we re-chain the same trained layers onto a fresh `Input` - that gives us a model with all the intermediate outputs.)

In [ ]:
# activation model: outputs after every conv / pool layer
inputs = keras.Input(shape=(150, 150, 3))
x = inputs
layer_outputs, layer_names = [], []
for layer in model.layers:
    x = layer(x)
    if layer.__class__.__name__ in ("Conv2D", "MaxPooling2D"):
        layer_outputs.append(x); layer_names.append(layer.name)
activation_model = keras.Model(inputs, layer_outputs)

activations = activation_model.predict(demo[None, ...], verbose=0)
for name, act in zip(layer_names, activations):
    print(f"{name:18s} {act.shape[1:]}")   # (height, width, num feature maps)

In [ ]:
def show_activations(act, layer_name, ncol=8, maxmaps=32):
    n = min(act.shape[-1], maxmaps); nrow = int(np.ceil(n / ncol))
    plt.figure(figsize=(ncol * 1.4, nrow * 1.4))
    for i in range(n):
        plt.subplot(nrow, ncol, i + 1)
        plt.imshow(act[0, :, :, i], cmap="viridis"); plt.axis("off")
    plt.suptitle(f"{layer_name}: {act.shape[-1]} feature maps, each {act.shape[1]}x{act.shape[2]}", fontsize=13)
    plt.tight_layout(); plt.show()

# FIRST conv layer - edge & outline detectors; you can still see the animal
show_activations(activations[0], layer_names[0])

Now a **deep** layer. Notice the difference: most feature maps are dark (that filter didn't fire), the ones that do are tiny and abstract, and you can no longer "see the dog." The network has distilled the picture down to a handful of high-level concepts - that's the *magic*, and it's also why the raw activations get hard to read.

In [ ]:
# a DEEP layer - sparse and abstract (high-level concepts, not a recognizable dog)
show_activations(activations[-2], layer_names[-2])

## 3) How did we get the final answer? Grad-CAM

Those deep activations feed the dense layers that output P(dog). **Grad-CAM** asks: for the class the model chose, which regions of the last conv layer mattered most? It weights each feature map by how much it pushed the prediction, sums them, and paints a heatmap - **red = "this is where I looked."**

In [ ]:
# Grad-CAM reads the LAST conv layer (re-chained, same Keras-3-safe trick)
def make_grad_model(model):
    last_conv = [l.name for l in model.layers if isinstance(l, keras.layers.Conv2D)][-1]
    inputs = keras.Input(shape=(150, 150, 3)); x = inputs; conv = None
    for layer in model.layers:
        x = layer(x)
        if layer.name == last_conv: conv = x
    return keras.Model(inputs, [conv, x]), last_conv

grad_model, last_conv = make_grad_model(model)
print("Grad-CAM reads:", last_conv)

def gradcam(img):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img[None, ...])
        channel = preds[:, 0]                       # the P(dog) score
    grads = tape.gradient(channel, conv_out)         # how each conv pixel moves the score
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))   # importance weight per feature map
    hm = tf.squeeze(conv_out[0] @ pooled[..., None]) # weighted sum of feature maps
    hm = tf.maximum(hm, 0) / (tf.reduce_max(hm) + 1e-8)
    return hm.numpy()

def show_decision(path, title):
    img = load(path)
    p = float(model.predict(img[None, ...], verbose=0)[0][0])
    guess = "dog" if p > 0.5 else "cat"
    hm = gradcam(img)
    hm_rgb = cm.get_cmap("jet")(hm)[..., :3]
    hm_big = np.array(Image.fromarray(np.uint8(255 * hm_rgb)).resize((150, 150), Image.BILINEAR)) / 255.
    overlay = 0.55 * img + 0.45 * hm_big
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
    ax[0].imshow(img); ax[0].set_title("what went in"); ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title(f"where it looked\nP(dog)={p:.2f}  ->  {guess}"); ax[1].axis("off")
    plt.suptitle(title, fontsize=13); plt.tight_layout(); plt.show()

### Find the most confident HIT and the most confident MISS

We scan a chunk of the test set, then pick the prediction the model was **most sure about and got right**, and the one it was **most sure about and got wrong**.

In [ ]:
files = ([("cat", f) for f in sorted(glob.glob(test_dir + "/cats/*.jpg"))[:300]] +
         [("dog", f) for f in sorted(glob.glob(test_dir + "/dogs/*.jpg"))[:300]])
X = np.stack([load(f) for _, f in files])
y_true = np.array([0 if lab == "cat" else 1 for lab, _ in files])
probs = model.predict(X, verbose=0).ravel()
pred = (probs > 0.5).astype(int)
conf = np.abs(probs - 0.5)              # distance from the 0.5 fence = confidence
correct = pred == y_true

hit  = np.where(correct)[0][np.argmax(conf[correct])]
miss = np.where(~correct)[0][np.argmax(conf[~correct])]
print("confident HIT :", os.path.basename(files[hit][1]),  "true", files[hit][0],  "P(dog)=%.3f" % probs[hit])
print("confident MISS:", os.path.basename(files[miss][1]), "true", files[miss][0], "P(dog)=%.3f" % probs[miss])

In [ ]:
# CONFIDENT HIT - right answer, and the heat should sit on the animal
show_decision(files[hit][1], "CONFIDENT HIT  -  right, and for a sensible reason")

In [ ]:
# CONFIDENT MISS - the model is very sure, and very wrong. Where did it look?
show_decision(files[miss][1], "CONFIDENT MISS  -  100% sure, 100% wrong")

## Takeaways

- **Feature maps** show the image being taken apart: edges and outlines early, sparse abstract concepts deep. Same picture, re-encoded layer by layer.
- **Grad-CAM** turns the final answer into a picture: *where the model looked*.
- On the **confident hit**, the heat lands on the animal - right answer, right reason.
- On the **confident miss**, watch where the heat goes. If it's on the **background** (a doghouse, grass, a couch) instead of the animal, the model learned a **spurious cue** - "outdoors, therefore dog." It can be confidently wrong for a reason that has nothing to do with the pet.
- That's why interpretability matters: a high accuracy number doesn't tell you *why* a model is right, and Grad-CAM is a quick way to catch a model that's cheating.